# Phase 3 — Classical ML Baselines
Water-System Anomaly Detection | Adaptive Temporal Transformer Project

This notebook trains and evaluates three classical ML baselines:
- Logistic Regression
- Random Forest
- XGBoost

All results are generated from actual model execution. No values are fabricated.

In [ ]:
import sys, json
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

from data_loader import load_raw
from preprocessing import run_preprocessing
from baselines import (
    build_tabular_features, feature_names,
    build_logistic_regression, build_random_forest, build_xgboost,
    train_model, predict_proba, save_model, select_threshold, load_model,
)
from evaluation import compute_metrics, measure_inference_time, curves
from config import TARGET_COL, SEED
import joblib

np.random.seed(SEED)
print('Imports OK')

## 1. Load Processed Data

In [ ]:
df = load_raw()
train_df, val_df, test_df, scaler, feat_cols = run_preprocessing(df, save=False)

X_train, y_train = build_tabular_features(train_df, feat_cols)
X_val,   y_val   = build_tabular_features(val_df,   feat_cols)
X_test,  y_test  = build_tabular_features(test_df,  feat_cols)
feat_names_list  = feature_names(feat_cols)

print(f'Train: {X_train.shape}, anomaly={y_train.mean()*100:.1f}%')
print(f'Val:   {X_val.shape},   anomaly={y_val.mean()*100:.1f}%')
print(f'Test:  {X_test.shape},  anomaly={y_test.mean()*100:.1f}%')
print(f'Features: {X_train.shape[1]}')

## 2. Class Imbalance Analysis

The dataset has ~13.4% anomaly rate overall. In the test set it is ~20.2%.
This imbalance means accuracy alone is misleading — a model predicting all-normal
would achieve ~80% accuracy on the test set while detecting zero anomalies.
We therefore focus on F1, Recall, Precision, and PR-AUC.

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
spw   = float(n_neg / n_pos)
print(f'Training set — Normal: {n_neg}, Anomaly: {n_pos}')
print(f'Imbalance ratio (neg/pos): {spw:.2f}')
print(f'Strategy: class_weight=balanced (LR, RF) | scale_pos_weight={spw:.2f} (XGBoost)')

## 3. Logistic Regression

In [ ]:
lr = build_logistic_regression()
lr, lr_time = train_model(lr, X_train, y_train)
print(f'Training time: {lr_time:.2f}s')

prob_val_lr  = predict_proba(lr, X_val)
prob_test_lr = predict_proba(lr, X_test)
thresh_lr    = select_threshold(y_val, prob_val_lr)
print(f'Threshold (val F1-max): {thresh_lr:.3f}')

pred_test_lr = (prob_test_lr >= thresh_lr).astype(int)
m_lr = compute_metrics(y_test, pred_test_lr, prob_test_lr)
print(f"Test F1={m_lr['f1']:.4f}  ROC-AUC={m_lr['roc_auc']:.4f}  PR-AUC={m_lr['pr_auc']:.4f}")
print(classification_report(y_test, pred_test_lr, target_names=['Normal','Anomaly']))

## 4. Random Forest

In [ ]:
rf = build_random_forest()
rf, rf_time = train_model(rf, X_train, y_train)
print(f'Training time: {rf_time:.2f}s')

prob_val_rf  = predict_proba(rf, X_val)
prob_test_rf = predict_proba(rf, X_test)
thresh_rf    = select_threshold(y_val, prob_val_rf)
print(f'Threshold (val F1-max): {thresh_rf:.3f}')

pred_test_rf = (prob_test_rf >= thresh_rf).astype(int)
m_rf = compute_metrics(y_test, pred_test_rf, prob_test_rf)
print(f"Test F1={m_rf['f1']:.4f}  ROC-AUC={m_rf['roc_auc']:.4f}  PR-AUC={m_rf['pr_auc']:.4f}")
print(classification_report(y_test, pred_test_rf, target_names=['Normal','Anomaly']))

## 5. XGBoost

In [ ]:
xgb_model = build_xgboost(scale_pos_weight=spw)
xgb_model, xgb_time = train_model(xgb_model, X_train, y_train, X_val, y_val)
print(f'Training time: {xgb_time:.2f}s')

prob_val_xgb  = predict_proba(xgb_model, X_val)
prob_test_xgb = predict_proba(xgb_model, X_test)
thresh_xgb    = select_threshold(y_val, prob_val_xgb)
print(f'Threshold (val F1-max): {thresh_xgb:.3f}')

pred_test_xgb = (prob_test_xgb >= thresh_xgb).astype(int)
m_xgb = compute_metrics(y_test, pred_test_xgb, prob_test_xgb)
print(f"Test F1={m_xgb['f1']:.4f}  ROC-AUC={m_xgb['roc_auc']:.4f}  PR-AUC={m_xgb['pr_auc']:.4f}")
print(classification_report(y_test, pred_test_xgb, target_names=['Normal','Anomaly']))

## 6. Load Saved Results and Display Comparison

In [ ]:
comp = pd.read_csv('../results/baselines/baseline_comparison.csv')
print(comp.to_string(index=False))

## 7. Visualisations (pre-generated)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, fname in zip(axes, ['roc_comparison.png', 'pr_comparison.png']):
    img = mpimg.imread(f'../figures/baselines/{fname}')
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, name in zip(axes, ['logistic_regression','random_forest','xgboost']):
    img = mpimg.imread(f'../figures/baselines/confusion_matrix_{name}.png')
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 8. Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, name in zip(axes, ['logistic_regression','random_forest','xgboost']):
    img = mpimg.imread(f'../figures/baselines/feature_importance_{name}.png')
    ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 9. Error Analysis

In [ ]:
err = pd.read_csv('../results/baselines/error_analysis.csv')
print('Error sample counts per model:')
print(err.groupby(['model','error_type']).size().unstack(fill_value=0))
print('\nSample FN rows (missed anomalies):')
print(err[err['error_type']=='FN'].head(6).to_string(index=False))

## 10. Interpretation

**Logistic Regression** achieved F1=0.885, ROC-AUC=0.963. It produced the highest FNR (0.154),
meaning it missed ~15% of anomalies. This is expected — LR is a linear model and cannot
capture the nonlinear sensor interactions that characterise anomaly events.

**Random Forest** achieved F1=0.978, ROC-AUC=0.997. It captured nonlinear feature interactions
effectively. The rolling statistics features (rmean, rstd) were among the most important,
confirming that temporal context is valuable even for classical models.

**XGBoost** achieved F1=0.978, ROC-AUC=0.998, with the lowest FNR (0.007). It trained
significantly faster than Random Forest (3.7s vs 66s) at comparable performance.

**Class imbalance** was handled via class weighting. Without this, all models would have
been biased toward predicting normal, inflating accuracy while missing anomalies.

**Temporal limitation**: Both RF and XGBoost rely on hand-crafted rolling statistics to
capture temporal context. They cannot learn arbitrary temporal dependencies the way an
LSTM or Transformer can. This is the key motivation for the deep learning phases.